# LeetCode 861 — Score After Flipping Matrix bằng Python

Link đề: https://leetcode.com/problems/score-after-flipping-matrix/ · Độ khó: **Medium** · Chủ đề: Greedy, Bit Manipulation, Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def matrixScore(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: **bit cao nhất quan trọng hơn mọi bit còn lại cộng lại**, nên trước hết lật hàng để cột đầu toàn `1`; sau đó mỗi cột còn lại độc lập: lật cột nếu số `0` nhiều hơn số `1`. Oracle: **vét cạn mọi tổ hợp lật hàng và cột** (`2^(m+n)`) trên ma trận nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận nhị phân `m × n`, mỗi lượt lật (đổi `0↔1`) một hàng hoặc một cột; mỗi hàng là một số nhị phân; điểm là tổng các hàng; tối đa hoá điểm.
2. Nắm công cụ Python: `int(s, 2)`, dịch bit `<<`, `XOR`, `zip(*grid)` (chuyển vị), `sum`, `max`, và biểu diễn mỗi hàng thành một số nguyên.
3. Đi từ **vét cạn tập hàng lật (`2^m`)** → **tham lam hai bước (hàng rồi cột) trên ma trận sửa tại chỗ** → **tham lam không sửa ma trận (đếm theo hàng gốc)** → **bitmask số nguyên**.
4. Hiểu mấu chốt: bit cao nhất của mỗi hàng có trọng số `2^(n−1)` lớn hơn tổng mọi bit thấp (`2^(n−1) > 2^(n−1) − 1`), nên **mọi hàng phải bắt đầu bằng `1`**; sau đó mỗi cột `j ≥ 1` chọn đa số.
5. Tránh lỗi kinh điển: lật cột đầu theo đa số (làm bit cao mất), lật cột trước hàng sau, đếm số `1` thay vì `max(1, 0)`, sai trọng số `2^(n−1−j)`, và sửa ma trận đầu vào khi đề không cho phép.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `grid` kích thước `m × n`.

Một **lượt** là chọn một hàng hoặc một cột bất kỳ rồi lật mọi giá trị trong đó (`0 → 1`, `1 → 0`).

Mỗi hàng được hiểu là một số nhị phân (bit đầu là bit cao nhất); **điểm** của ma trận là tổng các số đó.

Trả về điểm **cao nhất** sau một số lượt bất kỳ (kể cả không lượt nào).

**Ví dụ 1:** `grid = [[0,0,1,1],[1,0,1,0],[1,1,0,0]]` → `39` (`0b1111 + 0b1001 + 0b1111 = 15 + 9 + 15`).

**Ví dụ 2:** `grid = [[0]]` → `1`.

**Ràng buộc:** `1 <= m, n <= 20`; `grid[i][j]` là `0` hoặc `1`.

In [1]:
import itertools
import random
import time
from typing import List

ex1 = [[0, 0, 1, 1], [1, 0, 1, 0], [1, 1, 0, 0]]
ex2 = [[0]]
examples = [(ex1, 39), (ex2, 1)]
for g, want in examples:
    print(g, "->", want)

[[0, 0, 1, 1], [1, 0, 1, 0], [1, 1, 0, 0]] -> 39
[[0]] -> 1


## 3. Kiến thức nền

### 3.1 Hàng nhị phân thành số

Hàng `[1, 0, 1, 0]` là `0b1010 = 10`. Bit ở cột `j` (đếm từ 0, trái sang phải) có trọng số `2^(n−1−j)`. Dùng `int("".join(map(str, row)), 2)` hoặc `reduce` với dịch bit.

In [2]:
def row_value(row):
    v = 0
    for b in row:
        v = (v << 1) | b
    return v


print(row_value([1, 0, 1, 0]), int("1010", 2), [row_value(r) for r in ex1], sum(row_value(r) for r in ex1))

10 10 [3, 10, 12] 25


### 3.2 Lật bằng XOR

Lật một bit = `b ^ 1`. Lật cả hàng (với hàng là số nguyên `v` có `n` bit) = `v ^ ((1 << n) - 1)`. Lật cột `j` ảnh hưởng mỗi hàng ở bit `2^(n−1−j)`: `v ^ (1 << (n − 1 − j))`. **Thứ tự các lượt không quan trọng** và lật cùng hàng/cột hai lần là không làm gì, nên mỗi hàng/cột chỉ lật **0 hoặc 1 lần**: có `2^(m+n)` cấu hình.

In [3]:
n = 4
v = row_value([0, 0, 1, 1])
full = (1 << n) - 1
print(bin(v), "lật hàng ->", bin(v ^ full), "| lật cột 0 ->", bin(v ^ (1 << (n - 1))), "| lật cột 3 ->", bin(v ^ 1))

0b11 lật hàng -> 0b1100 | lật cột 0 -> 0b1011 | lật cột 3 -> 0b10


### 3.3 Trọng số bit cao áp đảo

Với `n` bit, bit cao nhất `2^(n−1)` **lớn hơn tổng** `2^(n−2) + … + 1 = 2^(n−1) − 1`. Do đó một hàng bắt đầu bằng `1` luôn lớn hơn mọi hàng bắt đầu bằng `0`, bất kể các bit sau.

In [4]:
n = 6
print("2^(n-1) =", 2 ** (n - 1), "> tổng các bit thấp =", sum(2 ** k for k in range(n - 1)))

2^(n-1) = 32 > tổng các bit thấp = 31


### 3.4 Chuyển vị bằng `zip(*grid)`

`zip(*grid)` cho từng **cột** (tuple). Hữu ích để đếm số `1` mỗi cột.

In [5]:
print([sum(col) for col in zip(*ex1)], list(zip(*ex1))[0])

[2, 1, 2, 1] (0, 1, 1)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1:

```
0 0 1 1
1 0 1 0
1 1 0 0
```

- Hàng 0 bắt đầu bằng `0`: lật hàng → `1 1 0 0`. Cột đầu giờ là `1,1,1`.
- Cột 1: các bit `1,0,1` → hai `1`, một `0` → giữ.
- Cột 2: `0,1,0` → hai `0` → lật cột → `1,0,1`.
- Cột 3: `0,0,0` → lật → `1,1,1`.

Kết quả: `1111, 1001, 1111` → `15 + 9 + 15 = 39` ✓.

Hai nhận xét: (1) cột đầu phải toàn `1` (vì trọng số bit đầu áp đảo); (2) sau khi cột đầu cố định, **lật cột bất kỳ** từ cột `1` trở đi **không phá** cột đầu, nên mỗi cột còn lại độc lập: chọn hướng có nhiều `1` hơn.

In [6]:
g = [r[:] for r in ex1]
for r in g:
    if r[0] == 0:
        r[:] = [1 - b for b in r]
print("sau khi lật hàng đưa cột đầu về 1:", g)
for j in range(1, 4):
    ones = sum(r[j] for r in g)
    print(f"cột {j}: {ones} số 1, {len(g) - ones} số 0 -> {'lật' if ones < len(g) - ones else 'giữ'}")

sau khi lật hàng đưa cột đầu về 1: [[1, 1, 0, 0], [1, 0, 1, 0], [1, 1, 0, 0]]
cột 1: 2 số 1, 1 số 0 -> giữ
cột 2: 1 số 1, 2 số 0 -> lật
cột 3: 0 số 1, 3 số 0 -> lật


## 5. Cách 1 — Brute Force: vét cạn tập hàng lật

**Ý tưởng:** thử mọi tập hàng lật (`2^m` mặt nạ). Với mỗi tập, mỗi cột độc lập: ta chọn lật hay không để có nhiều `1` hơn, đóng góp `max(ones, m − ones) · 2^(n−1−j)`. Lấy cực đại trên mọi mặt nạ.

(Không phải giả định "cột đầu toàn 1": brute này tự tìm mặt nạ tốt nhất, chỉ dùng sự thật hiển nhiên rằng cột lật độc lập khi hàng đã cố định.)

In [7]:
def score_rows_brute(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for mask in range(1 << m):                                   # bit i của mask = lật hàng i
        total = 0
        for j in range(n):
            ones = sum(grid[i][j] ^ ((mask >> i) & 1) for i in range(m))
            total += max(ones, m - ones) << (n - 1 - j)           # cột j: chọn hướng tốt hơn
        best = max(best, total)
    return best


print(score_rows_brute(ex1), score_rows_brute(ex2))
assert score_rows_brute(ex1) == 39 and score_rows_brute(ex2) == 1

39 1


In [8]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng 2^20 * 400 phép: có thể TLE trên test lớn)
class SolutionRowsBrute:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        best = 0
        for mask in range(1 << m):
            total = 0
            for j in range(n):
                ones = sum(grid[i][j] ^ ((mask >> i) & 1) for i in range(m))
                total += max(ones, m - ones) << (n - 1 - j)
            best = max(best, total)
        return best

**Độ phức tạp:** `O(2^m · m · n)`: với `m = 20` là `~4·10^8` phép, quá chậm. Bộ nhớ `O(1)`.

**Nút thắt:** ta thử mọi tập hàng lật trong khi chỉ cần **một** tập (làm cột đầu toàn `1`).

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam hai bước:**

1. **Hàng:** lật mọi hàng có bit đầu `= 0` để cột đầu toàn `1`. Lý do: bit đầu có trọng số `2^(n−1)` lớn hơn tổng mọi bit thấp, nên không bao giờ nên để hàng bắt đầu bằng `0` (đổi hàng đó sang bắt đầu bằng `1` luôn tăng giá trị hàng).
2. **Cột:** với mỗi cột `j ≥ 1`, đếm số `1`; nếu số `0` nhiều hơn thì lật cột (đạt `max(ones, zeros)`). Lật cột `j ≥ 1` không đổi cột đầu.

Tổng: `m · 2^(n−1) + Σ_{j ≥ 1} max(ones_j, m − ones_j) · 2^(n−1−j)`.

## 7. Các cách tiếp theo

Ngoài Cách 1, ta có thêm 3 cách: **Cách 2** tham lam sửa tại chỗ, **Cách 3** tham lam không sửa ma trận (đếm theo hàng gốc), **Cách 4** bitmask số nguyên.

### Cách 2 — Tham lam, sửa ma trận tại chỗ

**Giải thích từng đoạn:**

1. Với mỗi hàng có `grid[i][0] == 0`, lật cả hàng.
2. Với mỗi cột `j`, đếm số `1` (`ones`); nếu `ones < m - ones` thì coi như cột được lật: dùng `m - ones`.
3. Cộng `max(ones, m - ones) << (n - 1 - j)`.

In [9]:
def score_greedy_inplace(grid):
    g = [row[:] for row in grid]                                 # sao chép, không phá đầu vào
    m, n = len(g), len(g[0])
    for row in g:
        if row[0] == 0:                                          # bit cao nhất phải là 1
            for j in range(n):
                row[j] ^= 1
    total = 0
    for j in range(n):
        ones = sum(g[i][j] for i in range(m))
        total += max(ones, m - ones) << (n - 1 - j)              # cột: chọn hướng nhiều 1 hơn
    return total


print(score_greedy_inplace(ex1), score_greedy_inplace(ex2))
assert score_greedy_inplace(ex1) == 39 and score_greedy_inplace(ex2) == 1

39 1


In [10]:
# Code nộp LeetCode cho Cách 2 (sửa grid tại chỗ — LeetCode cho phép)
class SolutionInplace:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        for row in grid:
            if row[0] == 0:
                for j in range(n):
                    row[j] ^= 1
        total = 0
        for j in range(n):
            ones = sum(grid[i][j] for i in range(m))
            total += max(ones, m - ones) << (n - 1 - j)
        return total

**Độ phức tạp:** `O(m·n)` thời gian, `O(1)` thêm (nếu sửa tại chỗ).

### Cách 3 — Tham lam không sửa ma trận

**Ý tưởng:** hàng `i` được lật khi `grid[i][0] == 0`. Sau đó bit ở `(i, j)` là `grid[i][j] ^ (1 - grid[i][0])`... gọn hơn: bit sau khi lật hàng bằng **1 khi `grid[i][j] == grid[i][0]`** (vì hàng lật chỉ khi `grid[i][0] = 0`; khi đó bit mới `= 1 − grid[i][j]`, và `= 1` đúng khi `grid[i][j] = 0 = grid[i][0]`; hàng không lật: bit mới `= grid[i][j]`, `= 1` khi `grid[i][j] = 1 = grid[i][0]`). Do đó `ones_j = #{i : grid[i][j] == grid[i][0]}`. Cột `0` luôn `ones = m`.

In [11]:
def score_greedy_count(grid):
    m, n = len(grid), len(grid[0])
    total = m << (n - 1)                                         # cột 0 toàn 1
    for j in range(1, n):
        ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
        total += max(ones, m - ones) << (n - 1 - j)
    return total


print(score_greedy_count(ex1), score_greedy_count(ex2))
assert score_greedy_count(ex1) == 39 and score_greedy_count(ex2) == 1

39 1


In [12]:
# Code nộp LeetCode cho Cách 3 (không sửa đầu vào)
class SolutionCount:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        total = m << (n - 1)
        for j in range(1, n):
            ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
            total += max(ones, m - ones) << (n - 1 - j)
        return total

**Độ phức tạp:** `O(m·n)` thời gian, `O(1)` bộ nhớ, không phá đầu vào.

### Cách 4 — Bitmask số nguyên

**Ý tưởng:** mỗi hàng là một số nguyên `r`. Chuẩn hoá: nếu bit cao nhất bằng `0` thì `r ^= full`. Sau đó với mỗi cột `j` đếm `ones = Σ (r >> (n−1−j)) & 1`.

In [13]:
def score_bitmask(grid):
    m, n = len(grid), len(grid[0])
    full = (1 << n) - 1
    rows = []
    for row in grid:
        v = 0
        for b in row:
            v = (v << 1) | b
        if not (v >> (n - 1)) & 1:                               # bit cao nhất là 0: lật hàng
            v ^= full
        rows.append(v)
    total = 0
    for bit in range(n):                                         # bit 0 là cột cuối
        ones = sum((v >> bit) & 1 for v in rows)
        total += max(ones, m - ones) << bit
    return total


print(score_bitmask(ex1), score_bitmask(ex2))
assert score_bitmask(ex1) == 39 and score_bitmask(ex2) == 1

39 1


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionBitmask:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        full = (1 << n) - 1
        rows = []
        for row in grid:
            v = 0
            for b in row:
                v = (v << 1) | b
            if not (v >> (n - 1)) & 1:
                v ^= full
            rows.append(v)
        total = 0
        for bit in range(n):
            ones = sum((v >> bit) & 1 for v in rows)
            total += max(ones, m - ones) << bit
        return total

**Độ phức tạp:** `O(m·n)` thời gian, `O(m)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 3** với `grid = [[0,0,1,1],[1,0,1,0],[1,1,0,0]]`:

| cột `j` | các `grid[i][j] == grid[i][0]` | `ones` | `max(ones, 3 − ones)` | trọng số | đóng góp |
|---|---|---|---|---|---|
| 0 | luôn đúng | 3 | 3 | 8 | 24 |
| 1 | `0==0`, `0==1`?, `1==1` → đúng, sai, đúng | 2 | 2 | 4 | 8 |
| 2 | `1==0`?, `1==1`, `0==1`? → sai, đúng, sai | 1 | 2 | 2 | 4 |
| 3 | `1==0`?, `0==1`?, `0==1`? → sai, sai, sai | 0 | 3 | 1 | 3 |

Tổng `24 + 8 + 4 + 3 = 39` ✓. Hàm trace in từng bước:

In [15]:
def trace_count(grid):
    m, n = len(grid), len(grid[0])
    total = m << (n - 1)
    print(f"m = {m}, n = {n}")
    print(f"{'j':>2} {'ones':>5} {'zeros':>6} {'chọn':>5} {'trọng số':>9} {'đóng góp':>9}")
    print(f"{0:>2} {m:>5} {0:>6} {m:>5} {1 << (n - 1):>9} {m << (n - 1):>9}   (cột 0 toàn 1)")
    for j in range(1, n):
        ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
        pick = max(ones, m - ones)
        add = pick << (n - 1 - j)
        total += add
        print(f"{j:>2} {ones:>5} {m - ones:>6} {pick:>5} {1 << (n - 1 - j):>9} {add:>9}")
    print("=> tổng =", total)
    return total


assert trace_count(ex1) == 39

m = 3, n = 4
 j  ones  zeros  chọn  trọng số  đóng góp
 0     3      0     3         8        24   (cột 0 toàn 1)
 1     2      1     2         4         8
 2     1      2     2         2         4
 3     0      3     3         1         3
=> tổng = 39


**Trace Cách 2 (sửa tại chỗ)** cho thấy ma trận sau bước 1 (lật hàng), rồi từng cột:

In [16]:
def trace_inplace(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    for i, row in enumerate(g):
        if row[0] == 0:
            g[i] = [b ^ 1 for b in row]
            print(f"lật hàng {i}: {row} -> {g[i]}")
    print("sau bước 1:", g)
    total = 0
    for j in range(n):
        ones = sum(g[i][j] for i in range(m))
        flip = ones < m - ones
        take = max(ones, m - ones)
        total += take << (n - 1 - j)
        print(f"cột {j}: ones={ones}, {'lật' if flip else 'giữ'} -> {take} bit 1")
    print("=> tổng =", total)
    return total


assert trace_inplace(ex1) == 39

lật hàng 0: [0, 0, 1, 1] -> [1, 1, 0, 0]
sau bước 1: [[1, 1, 0, 0], [1, 0, 1, 0], [1, 1, 0, 0]]
cột 0: ones=3, giữ -> 3 bit 1
cột 1: ones=2, giữ -> 2 bit 1
cột 2: ones=1, lật -> 2 bit 1
cột 3: ones=0, lật -> 3 bit 1
=> tổng = 39


## 9. Code nộp LeetCode

Chọn **Cách 3** (không phá đầu vào, `O(mn)`). Giữ nguyên `class Solution`, `matrixScore(self, grid)`:

In [17]:
from typing import List


class Solution:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        total = m << (n - 1)                              # sau khi lật hàng, cột đầu toàn 1
        for j in range(1, n):
            # bit (i, j) sau khi lật hàng bằng 1 khi grid[i][j] == grid[i][0]
            ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
            total += max(ones, m - ones) << (n - 1 - j)   # cột j: lật nếu 0 nhiều hơn 1
        return total


print(Solution().matrixScore(ex1), Solution().matrixScore(ex2))      # 39 1
assert Solution().matrixScore(ex1) == 39 and Solution().matrixScore(ex2) == 1

39 1


## 10. Vì sao đúng

**(a) Mỗi hàng/cột lật tối đa một lần.** Lật hai lần là không làm gì và các lượt giao hoán (lật là XOR), nên cấu hình tối ưu là một cặp (tập hàng lật `R`, tập cột lật `C`). Bit `(i, j)` cuối là `grid[i][j] ^ r_i ^ c_j`.

**(b) Cột đầu toàn `1` trong lời giải tối ưu.** Giả sử một cấu hình có hàng `i` với bit đầu `0`. Lật hàng `i` đổi bit đầu thành `1` (tăng `2^(n−1)`) và đổi các bit sau (giảm tối đa `2^(n−1) − 1`). Giá trị hàng `i` tăng ít nhất `1`, các hàng khác không đổi: cấu hình không tối ưu. Vậy trong cấu hình tối ưu mọi hàng bắt đầu bằng `1`.

**(c) Sau đó các cột `j ≥ 1` độc lập.** Với `R` cố định để cột đầu toàn `1` ta có thể chọn `c_0` để cột đầu vẫn toàn `1`: bởi vì cột đầu hiện là `grid[i][0] ^ r_i ^ c_0 = 1` với mọi `i`, cố định `r_i = 1 ^ grid[i][0] ^ c_0`. Hai lựa chọn `c_0 ∈ {0, 1}` cho hai `R` bù nhau (lật hết mọi hàng), tương đương với lật toàn bộ ma trận — các cột `j ≥ 1` thấy cùng cấu trúc (bit lật toàn bộ thì cột `j` cũng đảo vai trò của `c_j`). Do đó có thể giả sử `c_0 = 0`.

**(d) Chọn `c_j`.** Đóng góp của cột `j` là `2^(n−1−j) · (số bit 1)`, trong đó số bit `1` là `ones_j` nếu `c_j = 0` hoặc `m − ones_j` nếu `c_j = 1`; chọn lớn hơn: `max(ones_j, m − ones_j)`.

**(e) Đếm `ones_j` mà không sửa ma trận** (Cách 3): xem phân tích ở mục 7.

Ta kiểm chứng bằng máy: oracle **vét cạn mọi cấu hình `(R, C)`** (`2^(m+n)`) cho mọi ma trận nhỏ (mục 12D) và ngẫu nhiên (12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn tập hàng | `O(2^m · m · n)` | `O(1)` | oracle một phần; TLE ở `m = 20` |
| 2. Tham lam sửa tại chỗ | `O(m·n)` | `O(1)` | đơn giản |
| 3. Tham lam đếm | `O(m·n)` | `O(1)` | **đáp án chính**, không phá input |
| 4. Bitmask | `O(m·n)` | `O(m)` | nhanh khi hàng là số nguyên |

Đo thực tế (`20 × 20`):

In [18]:
def rand_grid(rng, m, n):
    return [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]


def bench(fn, grid, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(grid)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(861)
G = rand_grid(rng, 20, 20)
for name, fn in [("inplace", score_greedy_inplace), ("count", score_greedy_count), ("bitmask", score_bitmask)]:
    print(f"20x20 {name:8s} {bench(fn, G) * 1e6:8.1f} us")
G2 = rand_grid(rng, 12, 20)
print(f"12x20 rows_brute (2^12 mặt nạ) {bench(score_rows_brute, G2, 1) * 1e3:8.1f} ms -> với m = 20 chậm ~256 lần")

20x20 inplace      30.1 us
20x20 count        28.6 us
20x20 bitmask      42.4 us


12x20 rows_brute (2^12 mặt nạ)    113.5 ms -> với m = 20 chậm ~256 lần


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi ma trận nhỏ với oracle `2^(m+n)`, (E) ngẫu nhiên có seed, tính chất, (F) giới hạn `20 × 20`.

### A. Ví dụ của đề

In [19]:
def oracle(grid):
    """Vét cạn MỌI cấu hình (tập hàng lật R, tập cột lật C): bit (i,j) = grid[i][j] ^ r_i ^ c_j. 2^(m+n) cấu hình."""
    m, n = len(grid), len(grid[0])
    full = (1 << n) - 1
    rows = [int("".join(map(str, r)), 2) for r in grid]
    best = 0
    for cmask in range(1 << n):                                  # cột lật: XOR vào mọi hàng
        for rmask in range(1 << m):
            total = 0
            for i in range(m):
                v = rows[i] ^ cmask
                if (rmask >> i) & 1:
                    v ^= full
                total += v
            if total > best:
                best = total
    return best


IMPLS = {
    "rows_brute": score_rows_brute,
    "inplace": score_greedy_inplace,
    "count": score_greedy_count,
    "bitmask": score_bitmask,
    "Solution": lambda g: Solution().matrixScore([r[:] for r in g]),
    "SolutionRowsBrute": lambda g: SolutionRowsBrute().matrixScore([r[:] for r in g]),
    "SolutionInplace": lambda g: SolutionInplace().matrixScore([r[:] for r in g]),
    "SolutionCount": lambda g: SolutionCount().matrixScore([r[:] for r in g]),
    "SolutionBitmask": lambda g: SolutionBitmask().matrixScore([r[:] for r in g]),
}


def check(grid, want=None, names=None):
    if want is None:
        want = oracle(grid)
    for name in (names or IMPLS):
        got = IMPLS[name]([r[:] for r in grid])
        assert got == want, (name, grid, got, want)


for g, want in examples:
    check(g, want)
    assert oracle(g) == want
print("ví dụ đề: oracle 2^(m+n) và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle 2^(m+n) và 9 cài đặt đúng


### B. Edge case

- `1 × 1`: `[[0]]` → `1`, `[[1]]` → `1`.
- Một hàng `1 × n`: lật hàng cho toàn `1` → `2^n − 1`.
- Một cột `m × 1`: mọi hàng thành `1` → `m`.
- Toàn `0` → mọi hàng toàn `1`: `m · (2^n − 1)`.
- Toàn `1` (đã tối ưu).
- Ma trận đã tối ưu (cột đầu toàn `1`, mỗi cột nhiều `1`).
- Cột `j ≥ 1` hoà nhau (`ones = m − ones`).

In [20]:
cases = [
    ([[0]], 1), ([[1]], 1),
    ([[0, 1, 0, 1]], 15), ([[0], [0], [1]], 3),
    ([[0, 0, 0], [0, 0, 0]], 14), ([[1, 1, 1], [1, 1, 1]], 14),
    ([[1, 0], [1, 1]], 5),                   # hoà ở cột 1? ones=1, zeros=1 -> max=1
    ([[1, 0, 0], [1, 1, 1], [1, 0, 1], [1, 1, 0]], None),
]
for g, want in cases:
    check(g, want)
assert Solution().matrixScore([[0, 0, 0], [0, 0, 0]]) == 2 * 7
assert Solution().matrixScore([[0, 1, 0, 1]]) == 15
print(len(cases), "edge case: oracle và mọi cài đặt cùng đúng")

8 edge case: oracle và mọi cài đặt cùng đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_rows_only(grid):                               # chỉ lật hàng, không lật cột
    total = 0
    for row in grid:
        v = row_value(row)
        if row[0] == 0:
            v ^= (1 << len(row)) - 1
        total += v
    return total


def bug_cols_first(grid):                              # lật cột theo đa số trước, rồi hàng
    m, n = len(grid), len(grid[0])
    g = [r[:] for r in grid]
    for j in range(n):
        if sum(g[i][j] for i in range(m)) < m - sum(g[i][j] for i in range(m)):
            for i in range(m):
                g[i][j] ^= 1
    total = 0
    for r in g:
        v = row_value(r)
        if r[0] == 0:
            v ^= (1 << n) - 1
        total += v
    return total


def bug_first_col_majority(grid):                      # coi cột đầu như mọi cột (chọn đa số), bỏ bước lật hàng
    m, n = len(grid), len(grid[0])
    total = 0
    for j in range(n):
        ones = sum(grid[i][j] for i in range(m))
        total += max(ones, m - ones) << (n - 1 - j)
    return total


def bug_count_ones_only(grid):                         # đếm số 1, không lật cột
    m, n = len(grid), len(grid[0])
    g = [r[:] for r in grid]
    for r in g:
        if r[0] == 0:
            r[:] = [b ^ 1 for b in r]
    return sum(sum(g[i][j] for i in range(m)) << (n - 1 - j) for j in range(n))


def bug_weight(grid):                                  # trọng số sai: 2^j thay vì 2^(n-1-j)
    m, n = len(grid), len(grid[0])
    g = [r[:] for r in grid]
    for r in g:
        if r[0] == 0:
            r[:] = [b ^ 1 for b in r]
    total = 0
    for j in range(n):
        ones = sum(g[i][j] for i in range(m))
        total += max(ones, m - ones) << j
    return total


BUGS = [bug_rows_only, bug_cols_first, bug_first_col_majority, bug_count_ones_only, bug_weight]
rng = random.Random(3)
tests = [rand_grid(rng, rng.randint(1, 4), rng.randint(2, 4)) for _ in range(400)]
for bug in BUGS:
    bad = next((g for g in tests if bug(g) != Solution().matrixScore(g)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {bad}: ra {bug(bad)}, đúng {Solution().matrixScore(bad)}")

bug_rows_only          sai trên [[0, 1, 1, 0], [0, 1, 1, 0]]: ra 18, đúng 30
bug_cols_first         sai trên [[1, 0], [0, 1], [1, 1]]: ra 7, đúng 8
bug_first_col_majority sai trên [[1, 1, 1, 0], [0, 0, 1, 0]]: ra 18, đúng 27
bug_count_ones_only    sai trên [[0, 1, 1, 0], [0, 1, 1, 0]]: ra 18, đúng 30
bug_weight             sai trên [[1, 1, 1, 0], [0, 0, 1, 0]]: ra 18, đúng 27


### D. Vét cạn mọi ma trận nhỏ

Mọi ma trận nhị phân `m × n` với `m, n ≥ 1` và `m·n ≤ 12`, `m + n ≤ 7`: `(m,n) ∈ {1×1..1×6, 2×1..2×5, 3×1..3×4, 4×1..4×3, 5×1..5×2, 6×1}`. Oracle `2^(m+n)` cấu hình mỗi ma trận.

In [22]:
t0 = time.perf_counter()
cnt = 0
shapes = [(m, n) for m in range(1, 7) for n in range(1, 7) if m * n <= 12 and m + n <= 7]
for m, n in shapes:
    for bits in range(1 << (m * n)):
        grid = [[(bits >> (i * n + j)) & 1 for j in range(n)] for i in range(m)]
        want = oracle(grid)
        for name in ("rows_brute", "inplace", "count", "bitmask", "Solution"):
            assert IMPLS[name](grid) == want, (name, grid)
        cnt += 1
print(len(shapes), "hình dạng,", cnt, "ma trận: 5 cài đặt khớp oracle 2^(m+n);", round(time.perf_counter() - t0, 1), "s")

21 hình dạng, 11658 ma trận: 5 cài đặt khớp oracle 2^(m+n); 1.0 s


### E. Ngẫu nhiên có seed và tính chất

Ma trận cỡ tới `5 × 5` vs oracle `2^(m+n)`; thêm tính chất: (1) **đầu vào không bị đổi** bởi `Solution`; (2) lật một hàng hoặc một cột của đầu vào **không đổi đáp án** (vì đó là một lượt hợp lệ).

In [23]:
rng = random.Random(20240861)
for _ in range(600):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = rand_grid(rng, m, n)
    check(g)
    snapshot = [r[:] for r in g]
    base = Solution().matrixScore(g)
    assert g == snapshot                                    # không phá đầu vào
    h = [r[:] for r in g]
    if rng.random() < 0.5:
        i = rng.randrange(m)
        h[i] = [b ^ 1 for b in h[i]]
    else:
        j = rng.randrange(n)
        for r in h:
            r[j] ^= 1
    assert Solution().matrixScore(h) == base               # lật là một lượt hợp lệ
print("600 ma trận ngẫu nhiên: khớp oracle, bảo toàn đầu vào, bất biến theo lượt lật")

600 ma trận ngẫu nhiên: khớp oracle, bảo toàn đầu vào, bất biến theo lượt lật


### F. Giới hạn `20 × 20`

Các cách tham lam khớp nhau; `rows_brute` thử với `m ≤ 12`; công thức đóng cho ma trận toàn `0`.

In [24]:
rng = random.Random(777)
fast = ["inplace", "count", "bitmask", "Solution", "SolutionInplace", "SolutionCount", "SolutionBitmask"]
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 20), rng.randint(1, 20))
    ref = score_greedy_count(g)
    for name in fast:
        assert IMPLS[name](g) == ref, name
    if len(g) <= 10:
        assert score_rows_brute(g) == ref
Z = [[0] * 20 for _ in range(20)]
assert Solution().matrixScore(Z) == 20 * (2 ** 20 - 1)
O = [[1] * 20 for _ in range(20)]
assert Solution().matrixScore(O) == 20 * (2 ** 20 - 1)
print("300 ma trận lớn: 7 cài đặt tham lam khớp nhau; toàn 0 hoặc toàn 1 cho", 20 * (2 ** 20 - 1))

300 ma trận lớn: 7 cài đặt tham lam khớp nhau; toàn 0 hoặc toàn 1 cho 20971500


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ lật hàng

Quên rằng lật **cột** có thể tăng tiếp.

In [25]:
g = [[1, 0, 0], [1, 0, 0], [1, 1, 0]]
print("chỉ lật hàng:", bug_rows_only(g), "| đúng:", Solution().matrixScore(g))
assert bug_rows_only(g) < Solution().matrixScore(g)

chỉ lật hàng: 14 | đúng: 19


**Vì sao sai:** cột `1` và `2` ở trên đa số là `0`: lật cột tăng điểm. **Đúng:** sau bước hàng, xét từng cột `j ≥ 1`.

### Lỗi 2: lật cột trước, hàng sau

Lật theo đa số ở cột đầu có thể **phá** ưu tiên bit cao.

In [26]:
bad = next(g for g in tests if bug_cols_first(g) != Solution().matrixScore(g))
print(bad, "cột trước:", bug_cols_first(bad), "| đúng:", Solution().matrixScore(bad))

[[1, 0], [0, 1], [1, 1]] cột trước: 7 | đúng: 8


**Vì sao sai:** thứ tự "cột rồi hàng" không dẫn tới cực đại vì quyết định cột phụ thuộc hàng. **Đúng:** hàng trước (đưa cột đầu về `1`), cột sau.

### Lỗi 3: xử lý cột đầu như các cột khác

Chọn đa số cho cột đầu và bỏ bước lật hàng.

In [27]:
g = [[0, 1], [0, 1], [1, 0]]
print("cột đầu theo đa số:", bug_first_col_majority(g), "| đúng:", Solution().matrixScore(g))
assert bug_first_col_majority(g) != Solution().matrixScore(g)

cột đầu theo đa số: 6 | đúng: 9


**Vì sao sai:** bit đầu của **mọi** hàng phải là `1` (lật hàng), không phải "đa số". **Đúng:** `total = m << (n - 1)` cho cột đầu.

### Lỗi 4: đếm số `1` mà quên lật cột

Chọn `ones` thay vì `max(ones, m − ones)`.

In [28]:
g = [[1, 0, 0], [1, 0, 0], [1, 0, 1]]
print("chỉ đếm 1:", bug_count_ones_only(g), "| đúng:", Solution().matrixScore(g))
assert bug_count_ones_only(g) < Solution().matrixScore(g)

chỉ đếm 1: 13 | đúng: 20


**Vì sao sai:** cột nhiều `0` nên lật cho nhiều `1` hơn. **Đúng:** `max(ones, m - ones)`.

### Lỗi 5: trọng số đảo

Bit trái là bit **cao**: trọng số `2^(n−1−j)`.

In [29]:
g = [[1, 0, 0], [1, 0, 1], [1, 0, 1]]
print("trọng số 2^j:", bug_weight(g), "| đúng:", Solution().matrixScore(g))
assert bug_weight(g) != Solution().matrixScore(g)

trọng số 2^j: 17 | đúng: 20


**Vì sao sai:** bit đầu tiên của hàng là MSB. **Đúng:** `<< (n - 1 - j)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tối đa hoá số nhị phân / tổng các số nhị phân bằng thao tác lật" → **tham lam theo bit cao nhất trước** (trọng số áp đảo), rồi tối ưu độc lập từng bit thấp hơn.

**Mẹo:**

1. Hỏi: bit cao có áp đảo tổng bit thấp không? (`2^k > 2^k − 1`: có.)
2. Các lượt lật giao hoán (XOR), mỗi hàng/cột tối đa một lần: không gian `2^(m+n)`.
3. Sau khi cố định cột đầu, các cột còn lại độc lập.
4. Đếm `ones` mà không cần sửa ma trận: so với cột đầu.
5. Oracle: vét cạn `2^(m+n)` trên ma trận nhỏ.

**Họ bài cùng pattern:** Flip Columns For Maximum Number of Equal Rows (1072), Maximum Binary String After Change (1702), Bulb Switcher (319), Cells with Odd Values in a Matrix (1252), Flip Game II (294).

## 15. Biến thể

1. **Điểm nhỏ nhất:** tối thiểu hoá tổng.
2. **Giới hạn số lượt `k`.**
3. **Cột đầu không lật** (chỉ được lật hàng).
4. **Tổng các bit 1 (không phải giá trị nhị phân).**
5. **Mỗi hàng đọc từ phải sang trái** (bit đầu là cuối hàng).

Cài biến thể 1 (điểm nhỏ nhất) và đối chiếu vét cạn:

In [30]:
def oracle_min(grid):
    m, n = len(grid), len(grid[0])
    full = (1 << n) - 1
    rows = [row_value(r) for r in grid]
    best = None
    for cmask in range(1 << n):
        for rmask in range(1 << m):
            total = 0
            for i in range(m):
                v = rows[i] ^ cmask
                if (rmask >> i) & 1:
                    v ^= full
                total += v
            best = total if best is None or total < best else best
    return best


def min_score(grid):
    """Đối xứng với bản max: ép bit đầu = 0 (mọi hàng), sau đó mỗi cột j >= 1 chọn bên ÍT số 1 hơn."""
    m, n = len(grid), len(grid[0])
    total = 0                                              # cột đầu toàn 0
    for j in range(1, n):
        ones = sum(1 for i in range(m) if grid[i][j] != grid[i][0])   # bit sau khi ép bit đầu = 0
        total += min(ones, m - ones) << (n - 1 - j)
    return total


cnt = 0
for m, n in [(1, 1), (1, 3), (2, 2), (2, 3), (3, 2), (3, 3), (2, 4), (4, 2)]:
    for bits in range(1 << (m * n)):
        grid = [[(bits >> (i * n + j)) & 1 for j in range(n)] for i in range(m)]
        assert min_score(grid) == oracle_min(grid), grid
        cnt += 1
print("điểm nhỏ nhất: công thức min khớp vét cạn trên", cnt, "ma trận; ví dụ 1 ->", min_score(ex1))

điểm nhỏ nhất: công thức min khớp vét cạn trên 1178 ma trận; ví dụ 1 -> 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập hàng | thử `2^m` mặt nạ, cột chọn tốt nhất | `O(2^m·mn)` | `O(1)` | oracle, `m` nhỏ |
| 2. Tham lam sửa tại chỗ | lật hàng, rồi `max(ones, zeros)` | `O(mn)` | `O(1)` | đơn giản |
| 3. Tham lam đếm | `ones = #{grid[i][j] == grid[i][0]}` | `O(mn)` | `O(1)` | **đáp án chính** |
| 4. Bitmask | hàng là số nguyên, XOR | `O(mn)` | `O(m)` | khi thích bit |

Cả bốn khớp oracle `2^(m+n)` trên mọi ma trận nhỏ và 600 ma trận ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().matrixScore([[0, 1], [1, 1]]), Solution().matrixScore([[0, 0], [0, 0]]), Solution().matrixScore([[1, 0, 1]]))

5 6 7


### Bài 2 (tính tay)

`grid = [[0,1,1],[1,0,0],[0,0,1]]`. (a) Hàng nào cần lật? (b) Sau đó cột nào lật? (c) Điểm tối đa? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện Cách 3:

In [32]:
def count_todo(grid):
    m, n = len(grid), len(grid[0])
    total = 0           # TODO 1: đóng góp của cột đầu (toàn 1)
    for j in range(1, n):
        ones = 0        # TODO 2: số hàng i có grid[i][j] == grid[i][0]
        total += 0      # TODO 3: max(ones, m - ones) dịch trái n - 1 - j
    return total


# Khi làm xong, bỏ comment để kiểm tra:
# assert count_todo(ex1) == 39 and count_todo(ex2) == 1

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **bitmask**:

In [33]:
class MySolution:
    def matrixScore(self, grid):
        pass            # TODO


# Khi làm xong:
# assert MySolution().matrixScore(ex1) == 39 and MySolution().matrixScore(ex2) == 1

### Bài 5 (mini challenge): chỉ được lật hàng

Viết `best_rows_only(grid)` — điểm lớn nhất nếu **chỉ** được lật hàng (không lật cột). So với vét cạn `2^m`.

In [34]:
def best_rows_only(grid):
    pass            # TODO


# Khi làm xong:
# assert best_rows_only([[0, 1], [1, 0]]) == 4

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: lật hàng bắt đầu bằng `0`.
- Mức 2: `[[0,1],[1,1]]` → `[[1,0],[1,1]]`; cột 1 có `0,1` hoà nên đóng góp `1`; cột đầu đóng góp `2·2 = 4`.
- Mức 3: kết quả `5 6 7`: `[[0,0],[0,0]]` → hai hàng `11` = `6`; `[[1,0,1]]` → `5` (một hàng không cần lật, nhưng cột 1 có `0` nhiều hơn nên lật cột 1 → `111 = 7`).

**Bài 2**

- Mức 1: hàng 0 và hàng 2 có bit đầu `0`.
- Mức 2: sau lật: `[[1,0,0],[1,0,0],[1,1,0]]`; cột 1: `ones = 1`, cột 2: `ones = 0`.
- Mức 3: lật cột 1 (`2` bit 1) và cột 2 (`3` bit 1): `3·4 + 2·2 + 3·1 = 19`.

**Bài 3**

- Mức 1: `total = m << (n - 1)`.
- Mức 2: `ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])`.
- Mức 3: `total += max(ones, m - ones) << (n - 1 - j)`.

**Bài 4**

- Mức 1: mỗi hàng thành số nguyên.
- Mức 2: nếu bit cao là `0`, XOR với `(1 << n) - 1`.
- Mức 3: đếm bit từng cột bằng `(v >> bit) & 1`.

**Bài 5**

- Mức 1: chỉ lật hàng: mỗi hàng độc lập.
- Mức 2: với mỗi hàng chọn `max(v, v ^ full)`.
- Mức 3: tổng các `max`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
assert Solution().matrixScore([[0, 1], [1, 1]]) == oracle([[0, 1], [1, 1]]) == 5
assert Solution().matrixScore([[0, 0], [0, 0]]) == 6 and Solution().matrixScore([[1, 0, 1]]) == 7
print("Bài 1 OK: 5 6 7")

Bài 1 OK: 5 6 7


### Bài 2

(a) Hàng 0 và hàng 2. (b) Cột 1 và cột 2. (c) `19`.

In [36]:
g = [[0, 1, 1], [1, 0, 0], [0, 0, 1]]
assert Solution().matrixScore(g) == oracle(g)
f = [[1, 0, 0], [1, 0, 0], [1, 1, 0]]
assert [sum(r[j] for r in f) for j in range(3)] == [3, 1, 0]
assert 3 * 4 + 2 * 2 + 3 * 1 == 19 == Solution().matrixScore(g)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [37]:
def count_answer(grid):
    m, n = len(grid), len(grid[0])
    total = m << (n - 1)
    for j in range(1, n):
        ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
        total += max(ones, m - ones) << (n - 1 - j)
    return total


assert count_answer(ex1) == 39 and count_answer(ex2) == 1
rng = random.Random(3)
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 5), rng.randint(1, 5))
    assert count_answer(g) == oracle(g)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def matrixScore(self, grid):
        m, n = len(grid), len(grid[0])
        full = (1 << n) - 1
        rows = []
        for row in grid:
            v = int("".join(map(str, row)), 2)
            rows.append(v ^ full if not v >> (n - 1) else v)
        return sum(max(c, m - c) << b for b in range(n) for c in [sum((v >> b) & 1 for v in rows)])


assert MySolutionAnswer().matrixScore(ex1) == 39 and MySolutionAnswer().matrixScore(ex2) == 1
rng = random.Random(4)
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 5), rng.randint(1, 5))
    assert MySolutionAnswer().matrixScore(g) == oracle(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def best_rows_only_answer(grid):
    n = len(grid[0])
    full = (1 << n) - 1
    return sum(max(v, v ^ full) for v in (row_value(r) for r in grid))


def best_rows_only_brute(grid):
    m, n = len(grid), len(grid[0])
    full = (1 << n) - 1
    rows = [row_value(r) for r in grid]
    return max(sum(rows[i] ^ (full if (mask >> i) & 1 else 0) for i in range(m)) for mask in range(1 << m))


assert best_rows_only_answer([[0, 1], [1, 0]]) == 4
rng = random.Random(5)
for _ in range(300):
    g = rand_grid(rng, rng.randint(1, 6), rng.randint(1, 6))
    assert best_rows_only_answer(g) == best_rows_only_brute(g) == bug_rows_only(g)
print("Bài 5 OK: chỉ lật hàng = tổng max(v, ~v) và trùng bản 'chỉ lật hàng' ở mục 12C")

Bài 5 OK: chỉ lật hàng = tổng max(v, ~v) và trùng bản 'chỉ lật hàng' ở mục 12C


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tối đa hoá tổng số nhị phân bằng lật hàng/cột → bit cao trước):**

```python
class Solution:
    def matrixScore(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        total = m << (n - 1)                                   # cột đầu toàn 1
        for j in range(1, n):
            ones = sum(1 for i in range(m) if grid[i][j] == grid[i][0])
            total += max(ones, m - ones) << (n - 1 - j)
        return total
```

**Checklist trước khi submit:**

- [ ] Lật hàng trước để **cột đầu toàn `1`** (bit cao áp đảo)?
- [ ] Từ cột `1` trở đi, mỗi cột lấy `max(ones, m − ones)`?
- [ ] Trọng số `2^(n−1−j)` (bit trái là MSB)?
- [ ] Đếm `ones` đúng sau khi lật hàng (so với `grid[i][0]`) hoặc sửa ma trận có chủ ý?
- [ ] Đã thử `[[0]]`, một hàng, một cột, toàn `0`, ví dụ 1 → `39`?
- [ ] Đúng tên `class Solution`, `matrixScore(self, grid)`, không in debug.